# Baseline V01

## Train Test

In [25]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [26]:
df = pd.read_csv("../data/processed/match_vector_v01.csv")

df.shape

(128, 60)

In [27]:
df["target"].value_counts()

target
 1    55
-1    45
 0    28
Name: count, dtype: int64

In [28]:
drop_cols = [
    "match_id",
    "match_date",
    "kick_off",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "competition_stage",
    "target",
    "year"
]

X = df.drop(columns=drop_cols)
y = df["target"]

X.shape, y.shape

((128, 50), (128,))

In [29]:
X.dtypes.value_counts()

float64    50
Name: count, dtype: int64

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((89, 50), (39, 50))

## Logistic Regression

In [31]:
log_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_log))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_log))

Accuracy: 0.3076923076923077

Confusion Matrix:
[[4 3 7]
 [0 2 6]
 [4 7 6]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.50      0.29      0.36        14
           0       0.17      0.25      0.20         8
           1       0.32      0.35      0.33        17

    accuracy                           0.31        39
   macro avg       0.33      0.30      0.30        39
weighted avg       0.35      0.31      0.32        39



## Random Forest

In [32]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.41025641025641024

Confusion Matrix:
[[ 4  5  5]
 [ 1  1  6]
 [ 2  4 11]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.57      0.29      0.38        14
           0       0.10      0.12      0.11         8
           1       0.50      0.65      0.56        17

    accuracy                           0.41        39
   macro avg       0.39      0.35      0.35        39
weighted avg       0.44      0.41      0.41        39



## Feature Importance

In [33]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

,feature,importance
4,home_Clearance,0.042021
12,home_Goal Keeper,0.039857
13,home_Interception,0.039048
29,away_Clearance,0.036138
19,home_Pressure,0.035553
3,home_Carry,0.034191
16,home_Pass,0.032974
32,away_Dribbled Past,0.031809
20,home_Shot,0.030999
0,home_Ball Receipt*,0.030641


## Prueba dummy

In [34]:
dummy_accuracy = y.value_counts(normalize=True).max()
dummy_accuracy

np.float64(0.4296875)

## Análisis

El Random Forest superó a la Dummy Baseline con la variable 'year' (0.48 vs 0.43). Cayó a 0.41 sin 'year'.

La Regresión Logística no logró converger y tuvo mal rendimiento

Los empates siguen siendo los más difíciles de predecir

El modelo captura mejor victorias locales y visitantes

Las variables más importantes suelen ser presión, pases y despejes.

### Limitaciones

Solo se utilizaron eventos agregados por equipo

No se incluyeron variables contextuales

No se incorporó información de jugadores

No se utilizaron los rankings de la FIFA ni Elo

No se realizaron optimizaciones ni hiperparámetros

El dataset contiene únicamente 128 partidos